<a href="https://colab.research.google.com/github/BLHmarwane/medical-gan-oct/blob/main/notebooks/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# CycleGAN + CBAM — Colab training notebook

Trains the OCT image-enhancement model on a Colab GPU.

## Before running

1. **Upload your dataset zip to Drive.** From your local project root:
   ```bash
   cd /path/to/medical-gan-oct
   zip -r processed.zip data/processed
   ```
   Then drag `processed.zip` into your Drive at `My Drive/processed.zip`.

2. **Enable GPU.** Runtime → Change runtime type → Hardware accelerator: **GPU**. Save.

## How storage is laid out

- **Data** is unzipped onto Colab's fast local disk (~10× faster reads than Drive).
- **Checkpoints and sample PNGs** are symlinked to Drive so they survive session shutdown.

Run cells top to bottom.


## 1. Verify GPU


In [1]:
!nvidia-smi

Sat Apr 25 23:17:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Clone the public repo and install


In [3]:
%cd /content
!rm -rf medical-gan-oct
!git clone https://github.com/BLHmarwane/medical-gan-oct.git
%cd medical-gan-oct
!pip install -q -r requirements.txt
!pip install -q -e .


/content
Cloning into 'medical-gan-oct'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: 'medical-gan-oct'
/content
fatal: not a git repository (or any of the parent directories): .git
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
ERROR: file:///content does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


## 4. Unzip the dataset to local Colab disk

Reads from Drive are slow (network mount). Unzipping to `/content/medical-gan-oct/data/processed/`
puts the dataset on Colab's fast local SSD. The zip's internal structure is
`data/processed/domain_A/...`, so we unzip at the repo root.


In [4]:
import shutil
from pathlib import Path

REPO = Path('/content/medical-gan-oct')
ZIP = Path('/content/drive/MyDrive/processed.zip')  # upload processed.zip here first

data_dir = REPO / 'data' / 'processed'
if data_dir.is_symlink():
    data_dir.unlink()
elif data_dir.exists():
    shutil.rmtree(data_dir)

assert ZIP.exists(), f'Missing {ZIP}. Upload processed.zip to Google Drive first.'
!unzip -q "{ZIP}" -d "{REPO}"
print('Dataset ready.')


Done.


## 5. Symlink checkpoints and logs to Drive

These get written DURING training; symlinking ensures they persist when the
Colab session ends. (Data does NOT get symlinked — it's read constantly
during training, so it stays on local disk for speed.)


In [5]:
import shutil
from pathlib import Path

REPO = Path('/content/medical-gan-oct')
DRIVE = Path('/content/drive/MyDrive/medical-gan-oct')

# Make sure Drive folders exist
(DRIVE / 'checkpoints').mkdir(parents=True, exist_ok=True)
(DRIVE / 'logs/samples').mkdir(parents=True, exist_ok=True)

def symlink(src: Path, dst: Path) -> None:
    if dst.is_symlink():
        dst.unlink()
    elif dst.exists():
        shutil.rmtree(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.symlink_to(src)

symlink(DRIVE / 'checkpoints', REPO / 'checkpoints')
symlink(DRIVE / 'logs',        REPO / 'logs')
print('Symlinks ready.')
!ls -la /content/medical-gan-oct/checkpoints /content/medical-gan-oct/logs

Symlinks ready.
lrwxrwxrwx 1 root root 50 Apr 25 23:18 /content/medical-gan-oct/checkpoints -> /content/drive/MyDrive/medical-gan-oct/checkpoints
lrwxrwxrwx 1 root root 43 Apr 25 23:18 /content/medical-gan-oct/logs -> /content/drive/MyDrive/medical-gan-oct/logs


## 6. Sanity-check the dataset


In [6]:
from pathlib import Path

root = Path('/content/medical-gan-oct/data/processed')
a = sorted((root / 'domain_A').glob('*'))
b = sorted((root / 'domain_B').glob('*'))
print(f'domain_A: {len(a)} files')
print(f'domain_B: {len(b)} files')
assert len(a) > 0 and len(b) > 0, 'Dataset is empty. Re-run the dataset unzip cell.'


domain_A: 100 files
domain_B: 100 files


## 7. Train

On T4: ~30s/epoch → ~100 minutes for 200 epochs.
Sample PNGs and checkpoints stream to Drive as they're written.


In [7]:
!python scripts/train.py --config configs/cyclegan_cbam_demo.yaml


python3: can't open file '/content/scripts/train.py': [Errno 2] No such file or directory


## 8. Preview samples and evaluate PSNR/SSIM


In [8]:
from pathlib import Path
from IPython.display import Image, display

samples = sorted(Path('/content/medical-gan-oct/logs/demo_samples').glob('epoch_*.png'))
if samples:
    print(f'{len(samples)} sample grids saved. Showing the latest:')
    display(Image(filename=str(samples[-1])))
else:
    print('No samples yet — training may still be starting.')

!python scripts/evaluate.py \
  --config configs/cyclegan_cbam_demo.yaml \
  --checkpoint checkpoints/demo/latest.pt \
  --limit 100 \
  --output logs/evaluation/demo_metrics.csv


No samples yet — training may still be starting.
